# CEH GEAR-Hourly: Zarr via Python + Xarray

**Launch this notebook:**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_ORG/YOUR_REPO/blob/main/gear_zarr_python.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/YOUR_ORG/YOUR_REPO/HEAD?labpath=gear_zarr_python.ipynb)
[![Open in DataLabs](https://img.shields.io/badge/Open%20in-NERC%20DataLabs-blue?logo=jupyter)](https://datalabs.nerc.ac.uk)

> Replace `YOUR_ORG/YOUR_REPO` with your GitHub path once published.

---

## What this notebook does

This notebook explores **GEAR-Hourly** (Gridded Estimates of Areal Rainfall — Hourly), a gridded rainfall dataset for Great Britain produced as part of the [FDRI (Floods and Droughts Research Infrastructure)](https://fdri.org.uk) project. 

Simple examples are shown for exploring and working with the dataset, which can be generalised to any gridded dataset stored on S3 storage, a storage medium that is becoming increasingly common for large gridded datasets. Datasets on S3 storage are typically stored in the Zarr format. However if you are familiar with working with NetCDF files with Xarray, this will be largely transparent to you with respect to the commands you can use to view and analyse the dataset.  

**We will:**
1. Show how to open a dataset stored in Zarr format 
2. Show how to extract and plot a time series at a single location
3. Show how to plot a map of rainfall at a single time step

**Data store:**
The data is currently publicly available as a trial through JASMIN object storage at the following URL:
```
https://fdri-o.s3-ext.jc.rl.ac.uk/gearhrly/gearhrly_15day_100km_chunks.zarr
```

| Runtime | Notes |
|---------|-------|
| Google Colab | ✅ Works out of the box — no installs needed |
| Binder | ✅ Works out of the box |
| NERC DataLabs | ✅ Login needed |
| JupyterLite | ✅ Should work with Pyodide kernel |
| Local Jupyter | ✅ Run `pip install xarray matplotlib` if needed |
| Quarto | ✅ Use `jupyter` engine with a Python kernel |

---

## 0. Install and import libraries

On most platforms `xarray` and `matplotlib` are already available.  
The install cell below is harmless to run even if they are already present.

In [1]:
# Install if needed (safe to run even if already installed)
%pip install -q xarray matplotlib zarr

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-expr 1.1.1 requires dask==2024.5.1, but you have dask 2024.5.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import xarray as xr
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

print(f"xarray  version: {xr.__version__}")

xarray  version: 2026.2.0


---
## 1. Open the Zarr store

`xr.open_zarr()` connects to the store over HTTPS without downloading any data yet.  
Data is only fetched when you actually access specific values. This behaviour is referred to as "Lazy Loading", and is important to be aware of when working with cloud-based datasets, as memory usage might not be what you expect.

For publically available data such as this, we do not need any credentials, and instead pass anon=True which means 'access the data as an anonymous user'.

In [2]:
ds = xr.open_zarr(
    "s3://gearhrly/gearhrly_15day_100km_chunks.zarr",
    storage_options={
        "anon": True,
        "endpoint_url": "https://fdri-o.s3-ext.jc.rl.ac.uk/"
    },
    consolidated=True
)

In [3]:
ds

<xarray.Dataset> Size: 5TB
Dimensions:          (time: 236688, y: 1251, x: 701, bnds: 2)
Coordinates:
  * time             (time) datetime64[ns] 2MB 1990-01-01 ... 2016-12-31T23:0...
  * y                (y) float64 10kB 1.25e+06 1.249e+06 1.248e+06 ... 1e+03 0.0
  * x                (x) float64 6kB 0.0 1e+03 2e+03 ... 6.98e+05 6.99e+05 7e+05
    lat              (y, x) float64 7MB dask.array<chunksize=(100, 100), meta=np.ndarray>
    lon              (y, x) float64 7MB dask.array<chunksize=(100, 100), meta=np.ndarray>
    time_bnds        (time, bnds) datetime64[ns] 4MB dask.array<chunksize=(360, 2), meta=np.ndarray>
    x_bnds           (x, bnds) float64 11kB dask.array<chunksize=(100, 2), meta=np.ndarray>
    y_bnds           (y, bnds) float64 20kB dask.array<chunksize=(100, 2), meta=np.ndarray>
    crs              int16 2B ...
Dimensions without coordinates: bnds
Data variables:
    min_dist         (time, y, x) float64 2TB dask.array<chunksize=(360, 100, 100), meta=np.ndarray>
    rainfall_amount  (time, y, x) float64 2TB dask.array<chunksize=(360, 100, 100), meta=np.ndarray>
    stat_disag       (time, y, x) float64 2TB dask.array<chunksize=(360, 100, 100), meta=np.ndarray>
Attributes: (12/29)
    Conventions:                   CF-1.6
    acknowledgement:               This research forms part of the SINATRA pr...
    cdm_data_type:                 Grid
    contributor_name:              Lewis, E., Quinn, N., Blenkinsop, S., Fowl...
    creator_email:                 enquiries@ceh.ac.uk
    creator_institution:           UK Centre for Ecology & Hydrology (UKCEH)
    ...                            ...
    standard_name_vocabulary:      CF Standard Name Table v70, http://cfconve...
    summary:                       The CEH-GEAR1hr-v2 dataset contains 1-km g...
    time_coverage_duration:        P27Y
    time_coverage_resolution:      P1H
    title:                         Gridded estimates of hourly areal rainfall...
    version:                       v2

Size of dataset in TeraBytes:

In [6]:
ds.nbytes/1E12

4.98155043749

The variables in the dataset are split into those that describe the coordinates, and those of the main data. We can see the three main data variables: 'min_dist', 'rainfall_amount' and 'stat_disag', you can click the page icon at the end of each variable's row to find out a little more about each.

The names in brackets next to the variable names (the second column of information) show you the dimensions that each variable is on. Here, all our data variables are on a 3D grid of time, y and x.

You can see in the Coordinates section that we have other coordinates besides those for the time, y and x dimensions. The lat and lon variables tell you the latitude and longitude conversion for each x,y gridpoint and the 'xxx_bnds' variables tell you the extent/valid range of each datapoint. So for example a gridpoint of (6000,6000) on a grid with a resolution of (1000,1000) would have extents/boundaries of (5500,6500) for both x and y, describing the extent of the gridbox this gridpoint represents. The same concept can be extended to the time-dimension too.

---
## 2. Time series at a single location

If you're familiar working with netcdf data in xarray, this will be very familiar.

Xarray allows you to select out a location using the dataset coordinates, instead of having to worry about indexing. A particularly powerful feature is that you don't even have to know the exact coordinates and can instead use the 'nearest' method, which finds the nearest gridpoint to your specified coordinates. 

You can use any of the coordinate variables in the dataset, not just those that the data is actually on. For the UKCEH-GEAR dataset it is on the x/y coordinates of the OS National Grid, but you may prefer to select out points based on longitude and latitude. All FDRI datasets, and in fact all datasets that adhere to the CF-Conventions, will also have longitude and latitude coordinates pre-calculated if the data is not already on a lon/lat grid, so these can also be used to select data points, and they are automatically and transparently mapped back to the underlying x/y coordinates. Look at the 'Coordinates' section in the view of the dataset metadata in Section 1 to see what coordinates are available.

Note that there's a line that explicitly loads the data into memory with the .compute() command. Remember Xarray lazy-loads data, and would normally only load in the data when the plot line is called. Normally this isn't a problem, but when reading data from the cloud the data is read in parallel with a library called dask in order to speed up the otherwise slow latency (lag time) when reading data over a HTTP connection. Dask introduces the concept of persistence *in addition to* lazy loading. Even when xarray has actually loaded the data, the dask layer beneath this *does not persist* the data, meaning that it is not available in-memory for re-use after plotting - or any other analysis - and would have to be retrieved again if you wanted to do anything further with the plotted data. This can be very slow and resource-intensive and is best avoided.

PUT INTO EXPANDABLE SECTION OR COLOURED "MORE INFORMATION" BOX
More information on lazy loading and persistence with Xarray and Dask.
Before we go any further it is very important to know that Xarray loads data lazily. This means that the actual data is never loaded or computed until it absolutely has to be. So you may run some code that computes the mean of the dataset (e.g. dmean = ds['variable'].mean(dim='time')), but the computation will only actually be carried out when you want to see the result of this calculation, e.g. through a plot or print statement, or saving to disk.

Dask also takes this a step further. Dask is a library designed to automatically parallelise computations; parallelisation is essential when dealing with chunked data such as Zarr on the cloud. Dask sits in the background working out how best to parallelise your computations, then whenever Xarray actually triggers a compute, Dask will automatically kick in and process the computation in parallel. However, with Dask, the output of the computation will not persist in memory, even if you save the output to a variable (e.g. dmean = ds['variable'].mean(dim='time')). If you run the code again, Dask will recompute everything from scratch, the actual data is not stored in the variable dmean, rather the Dask-instructions for computing it. This can make code very slow if not handled correctly. The best way to handle this is to make use of Dask's .persist()and .compute() methods. Appending this on to the end of a calculation, such as dmean = ds['variable'].mean(dim='time').compute() will keep the data in memory.

The main difference between persist and compute is that persist will allow you to continue coding whilst dask computes the result in the background, whereas compute will wait until the computation is complete before letting you continue coding. I tend to find compute behaves the most intuitively.

Now with that out the way let's get to actually looking at the data!

In [14]:
# Choose a location — e.g. Edinburgh, Scotland
# Adjust to any point within the dataset's extent
target_lon = -3.19
target_lat = 55.95

# The variable name — see the 'Data variables' of the view of the dataset metadata in Section 1 
var_name = "rainfall_amount"

# Select the nearest grid cell
# The coordinate names depend on the store — common names: longitude/latitude, lon/lat, x/y
point = ds[var_name].sel(lon=target_lon, lat=target_lat, method="nearest")

# Load the time series into memory, and persist it!
with ProgressBar():
    ts = point.compute()

ValueError: Could not automatically create PandasIndex for coord 'lon' with 2 dimensions. Please explicitly set the index using `set_xindex`.

Once you've selected the data you're interested in, a simple plot can be produced with:

In [ ]:
ts.plot()

Or you can customise the plot using matplotlib, the standard python plotting library, but this requires a few more lines of code to setup the plot and tweak the things you might want to:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4)) # customize the size of the plot

# plot the data on the axes (ax) we just created using ax.plot(x values, y values, customisations...)
# customize the line colour, width and transparency (alpha)
ax.plot(ts.time, ts.values, color="black", linewidth=0.8, alpha=0.85)

ax.set_title("GEAR-Hourly rainfall", fontsize=13)
ax.set_xlabel("Date")
ax.set_ylabel("")
ax.text(0.01, 0.97, f"lon={actual_lon:.3f}°, lat={actual_lat:.3f}°",
        transform=ax.transAxes, va="top", fontsize=9, color="grey")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

---
## 3. Map at a single time step

Select one time step and load the full spatial grid. xarray's `.plot()` creates a quick map from any 2-D DataArray.

In [ ]:
# Select the first time step
# You can use isel(time=N) for index-based selection,
# or sel(time="YYYY-MM-DD") for date-based selection
t_index = 0
slice_2d = ds[var_name].isel(time=t_index).load()

time_label = str(slice_2d.time.values)[:16]   # trim to minute precision
print(f"Time step : {time_label}")
print(f"Grid shape: {slice_2d.shape}")
print(f"Value range: {float(slice_2d.min()):.3f} to {float(slice_2d.max()):.3f} {units}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))

# xarray's built-in .plot() handles axis labels and colorbars automatically
slice_2d.plot(
    ax=ax,
    cmap="YlGnBu",
    cbar_kwargs={"label": f"{long_name} ({units})", "shrink": 0.7}
)

ax.set_title(f"GEAR-Hourly: {long_name}\n{time_label}", fontsize=12, fontweight="bold")
ax.set_aspect("equal")   # preserve grid cell shape
fig.tight_layout()
plt.show()

---
## 4. Accumulated rainfall over a time window

Sum the first 24 time steps (one day of hourly data) over the full grid.  
xarray handles the aggregation in one line — only the needed chunks are fetched.

In [ ]:
n_steps = min(24, len(ds.time))

total_24h = ds[var_name].isel(time=slice(0, n_steps)).sum(dim="time").load()

t_start = str(ds.time.values[0])[:16]
t_end   = str(ds.time.values[n_steps - 1])[:16]

fig, ax = plt.subplots(figsize=(6, 8))

total_24h.plot(
    ax=ax,
    cmap="YlGnBu",
    cbar_kwargs={"label": f"{n_steps}-hour total ({units})", "shrink": 0.7}
)

ax.set_title(
    f"GEAR-Hourly: {n_steps}-hour accumulated rainfall\n{t_start} to {t_end}",
    fontsize=12, fontweight="bold"
)
ax.set_aspect("equal")
fig.tight_layout()
plt.show()

---
## Tips

| Task | Code |
|------|------|
| Select by date | `ds[var].sel(time="2015-01-01T06:00")` |
| Select a date range | `ds[var].sel(time=slice("2015-01-01", "2015-01-31"))` |
| Monthly mean | `ds[var].resample(time="1ME").mean()` |
| Spatial mean over GB | `ds[var].mean(dim=["latitude", "longitude"])` |
| Save to NetCDF | `ds.to_netcdf("output.nc")` |
| List coordinate names | `list(ds.coords)` |

**Data citation:** CEH-GEAR, UKCEH. https://doi.org/10.5285/dbf13dd5-90cd-457a-a986-f2f9dd97e93c